# silver layer - data cleaning and anomaly detection

this notebook reads the raw data from the bronze layer, cleans invalid records,
joins household information and detects different types of anomalies before
saving the processed data into the silver layer.

In [0]:
# importing the spark functions needed for cleaning and analysis

from pyspark.sql import functions as f
from pyspark.sql.window import Window

In [0]:
# path of the household info file

household_info_path = "/Volumes/smart_meter_project_catalog/default/meter_problem_project_volume/household_info.csv"

In [0]:
# reading the bronze delta table

meter_df = spark.table("smart_meter_project_catalog.bronze.meter_readings")

# loading household information

household_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(household_info_path)
)

# taking a quick look before starting any transformations
# checking both schemas

meter_df.printSchema()

household_df.printSchema()
display(meter_df)

display(household_df)

root
 |-- meter_id: string (nullable = true)
 |-- household_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- units_consumed: double (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)

root
 |-- household_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- house_type: string (nullable = true)
 |-- avg_daily_consumption: integer (nullable = true)



meter_id,household_id,timestamp,units_consumed,ingestion_time,ingestion_date
M001,H001,2026-04-01T00:00:00.000Z,0.41,2026-07-10T18:19:52.606Z,2026-07-10
M002,H002,2026-04-01T00:00:00.000Z,0.1,2026-07-10T18:19:52.606Z,2026-07-10
M003,H003,2026-04-01T00:00:00.000Z,0.44,2026-07-10T18:19:52.606Z,2026-07-10
M004,H004,2026-04-01T00:00:00.000Z,0.23,2026-07-10T18:19:52.606Z,2026-07-10
M005,H005,2026-04-01T00:00:00.000Z,6.93,2026-07-10T18:19:52.606Z,2026-07-10
M006,H006,2026-04-01T00:00:00.000Z,0.8,2026-07-10T18:19:52.606Z,2026-07-10
M007,H007,2026-04-01T00:00:00.000Z,0.47,2026-07-10T18:19:52.606Z,2026-07-10
M008,H008,2026-04-01T00:00:00.000Z,0.2,2026-07-10T18:19:52.606Z,2026-07-10
M009,H009,2026-04-01T00:00:00.000Z,0.25,2026-07-10T18:19:52.606Z,2026-07-10
M010,H010,2026-04-01T00:00:00.000Z,0.25,2026-07-10T18:19:52.606Z,2026-07-10


household_id,city,house_type,avg_daily_consumption
H001,Jaipur,Independent,8
H002,Kolkata,Independent,10
H003,Jaipur,Villa,12
H004,Jaipur,Apartment,15
H005,Kolkata,Apartment,7
H006,Mumbai,Studio,9
H007,Pune,Studio,11
H008,Delhi,Studio,14
H009,Pune,Independent,6
H010,Hyderabad,Independent,13


In [0]:
# checking missing values in each column

null_summary = meter_df.select([
    f.count(
        f.when(f.col(c).isNull(), c)
    ).alias(c)
    for c in meter_df.columns
])

display(null_summary)

meter_id,household_id,timestamp,units_consumed,ingestion_time,ingestion_date
0,0,0,20,0,0


In [0]:
#checking any duplicate row not duplicate values in column because there are chances
# that some duplicate values can exist because they creating meaning to other data in other columns
#if it returns zero rows means no duplicate row is there 

duplicate_df = (
    meter_df
    .groupBy(
        "meter_id",
        "timestamp"
    )
    .count()
    .filter(
        f.col("count") > 1
    )
)

display(duplicate_df)

meter_id,timestamp,count


In [0]:
# converting timestamp into spark timestamp format

meter_df = meter_df.withColumn(
    "timestamp",
    f.to_timestamp("timestamp")
)

# no need to remove duplicates because no duplicate rows are found earlier

# keeping track of rows where the reading was missing

meter_df = meter_df.withColumn(
    "missing_units",
    f.col("units_consumed").isNull()
)


In [0]:
# replacing missing readings with the average consumption
# this avoids treating missing data as zero usage

average_units = meter_df.select(
    f.avg("units_consumed")
).first()[0]

meter_df = meter_df.fillna({
    "units_consumed": average_units
})

In [0]:
# checking whether the consumption value is valid

meter_df = meter_df.withColumn(
    "is_valid",
    f.when(
        f.col("units_consumed") >= 0,
        True
    ).otherwise(False)
)

# keeping only valid records for further analysis

meter_df = meter_df.filter(
    f.col("is_valid") == True
)

In [0]:
# joining household information

silver_df = (
    meter_df.join(
        household_df,
        on="household_id",
        how="left"
    )
)

display(silver_df)

household_id,meter_id,timestamp,units_consumed,ingestion_time,ingestion_date,missing_units,is_valid,city,house_type,avg_daily_consumption
H001,M001,2026-04-01T00:00:00.000Z,0.41,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8
H002,M002,2026-04-01T00:00:00.000Z,0.1,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Kolkata,Independent,10
H003,M003,2026-04-01T00:00:00.000Z,0.44,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Villa,12
H004,M004,2026-04-01T00:00:00.000Z,0.23,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Apartment,15
H005,M005,2026-04-01T00:00:00.000Z,6.93,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Kolkata,Apartment,7
H006,M006,2026-04-01T00:00:00.000Z,0.8,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Mumbai,Studio,9
H007,M007,2026-04-01T00:00:00.000Z,0.47,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Pune,Studio,11
H008,M008,2026-04-01T00:00:00.000Z,0.2,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Delhi,Studio,14
H009,M009,2026-04-01T00:00:00.000Z,0.25,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Pune,Independent,6
H010,M010,2026-04-01T00:00:00.000Z,0.25,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Hyderabad,Independent,13


In [0]:
# creating a window for each meter ordered by timestamp

meter_window = (
    Window
    .partitionBy("meter_id")
    .orderBy("timestamp")
)

In [0]:
# calculating the rolling average of the previous 3 readings

rolling_window = (
    meter_window
    .rowsBetween(-24, -1)
)

silver_df = (
    silver_df
    .withColumn(
        "rolling_avg_units",
        f.avg("units_consumed").over(rolling_window)
    )
)

# Detecting if any huge Spike occurs and outages creating a seperate column for each 

In [0]:
# marking readings that are much higher than the recent average

silver_df = (
    silver_df
    .withColumn(
        "is_spike",
        f.when(
            (f.col("rolling_avg_units").isNotNull()) &
            (f.col("units_consumed") > 3 * f.col("rolling_avg_units")),
            True
        ).otherwise(False)
    )
)

In [0]:
# readings with zero consumption may indicate a possible outage

silver_df = (
    silver_df
    .withColumn(
        "possible_outage",
        f.when(
            f.col("units_consumed") == 0,
            True
        ).otherwise(False)
    )
)

In [0]:
# calculating the average consumption for every meter

silver_df = (
    silver_df
    .withColumn(
        "meter_mean",
        f.avg("units_consumed").over(
            Window.partitionBy("meter_id")
        )
    )
)


# calculating the standard deviation for every meter

silver_df = (
    silver_df
    .withColumn(
        "meter_stddev",
        f.stddev("units_consumed").over(
            Window.partitionBy("meter_id")
        )
    )
)

In [0]:
# flagging readings that fall outside the normal range

silver_df = (
    silver_df
    .withColumn(
        "high_deviation",
        f.when(
            (
                f.col("units_consumed")
                >
                (
                    f.col("meter_mean")
                    + (2 * f.col("meter_stddev"))
                )
            ),
            True
        ).otherwise(False)
    )
)

In [0]:
# checking the anomaly detection results

# checking the final silver dataset

display(silver_df)

print(f"total records : {silver_df.count()}")

silver_df.printSchema()

household_id,meter_id,timestamp,units_consumed,ingestion_time,ingestion_date,missing_units,is_valid,city,house_type,avg_daily_consumption,rolling_avg_units,is_spike,possible_outage,meter_mean,meter_stddev,high_deviation
H001,M001,2026-04-01T00:00:00.000Z,0.41,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,null,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T01:00:00.000Z,0.8,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.41,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T02:00:00.000Z,0.41,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.605,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T03:00:00.000Z,0.96,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.5399999999999999,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T04:00:00.000Z,0.72,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.645,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T05:00:00.000Z,2.38,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.6599999999999999,true,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T06:00:00.000Z,4.82,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.9466666666666667,true,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T07:00:00.000Z,4.16,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,1.5,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T08:00:00.000Z,4.21,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,1.8325,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T09:00:00.000Z,3.87,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,2.0966666666666667,false,false,2.8427111801242226,1.9053857837008945,false


total records : 1400
root
 |-- household_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- units_consumed: double (nullable = false)
 |-- ingestion_time: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- missing_units: boolean (nullable = false)
 |-- is_valid: boolean (nullable = false)
 |-- city: string (nullable = true)
 |-- house_type: string (nullable = true)
 |-- avg_daily_consumption: integer (nullable = true)
 |-- rolling_avg_units: double (nullable = true)
 |-- is_spike: boolean (nullable = false)
 |-- possible_outage: boolean (nullable = false)
 |-- meter_mean: double (nullable = true)
 |-- meter_stddev: double (nullable = true)
 |-- high_deviation: boolean (nullable = false)



In [0]:
# checking how many anomalies were detected

display(

    silver_df.select(

        f.sum(f.col("is_spike").cast("int")).alias("total_spikes"),

        f.sum(f.col("possible_outage").cast("int")).alias("possible_outages"),

        f.sum(f.col("high_deviation").cast("int")).alias("high_deviation")

    )

)

total_spikes,possible_outages,high_deviation
44,29,21


In [0]:


# switching to the project catalog

spark.sql("""
use catalog smart_meter_project_catalog
""")

# creating the silver schema if it doesn't already exist

spark.sql("""
create schema if not exists silver
""")

DataFrame[]

In [0]:
# storing the cleaned and enriched data in the silver layer

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("silver.smart_meter_readings")
)

In [0]:
# reading the saved silver table

display(
    spark.table("silver.smart_meter_readings")
)

household_id,meter_id,timestamp,units_consumed,ingestion_time,ingestion_date,missing_units,is_valid,city,house_type,avg_daily_consumption,rolling_avg_units,is_spike,possible_outage,meter_mean,meter_stddev,high_deviation
H001,M001,2026-04-01T00:00:00.000Z,0.41,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,null,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T01:00:00.000Z,0.8,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.41,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T02:00:00.000Z,0.41,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.605,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T03:00:00.000Z,0.96,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.5399999999999999,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T04:00:00.000Z,0.72,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.645,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T05:00:00.000Z,2.38,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.6599999999999999,true,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T06:00:00.000Z,4.82,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,0.9466666666666667,true,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T07:00:00.000Z,4.16,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,1.5,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T08:00:00.000Z,4.21,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,1.8325,false,false,2.8427111801242226,1.9053857837008945,false
H001,M001,2026-04-01T09:00:00.000Z,3.87,2026-07-10T18:19:52.606Z,2026-07-10,false,true,Jaipur,Independent,8,2.0966666666666667,false,false,2.8427111801242226,1.9053857837008945,false
